In [1]:
import pandas as pd  
from pandas import Series, DataFrame 
import uproot 
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import comb
from scipy.stats import chi2
from scipy.special import comb
from scipy.optimize import lsq_linear
import sys
from plot_tools import *
from customStats import *
#import tools
import common_tools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
# from selection_cuts import selection_nominal
import mplhep as hep
from sklearn.model_selection import train_test_split
plt.style.use(hep.style.CMS)
plt.rcParams['figure.figsize'] = [10,8]
plt.rcParams['font.size'] = 24
plt.figure()
plt.close()
plt.rcParams.update({'figure.figsize':[10,8]})
plt.rcParams.update({'font.size':24})
import tensorflow as tf
import math
import zfit
from zfit import z
import xgboost as xgb
from scipy.interpolate import make_interp_spline
# from loadCutXGB import load_and_cutXGBclfs
from scipy.special import comb
from scipy.optimize import lsq_linear
zfit.settings.set_verbosity(0)
import json
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Oculta los mensajes de INFO y WARNING
from utils_efficiency import *
from utils_fits import*

2026-03-16 23:08:44.376025: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 23:08:44.403377: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-16 23:08:44.864432: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ghcp/miniconda3/envs/haza_wokr_env/lib/python3.8/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress th

# FUNCTIONS FOR CALCULATING EFFICIENCY

#  CODE FOR EFFICIENCY CALCULATION

In [2]:
import uproot
import pandas as pd

# --- RUTAS DE ARCHIVOS ---
f_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged.root"
f_gen_filtered = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged_Filtered.root"
f_reco_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/RecoGenV2_Angular_Merged.root"  
x_gboost_cut = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/BdtoK0smumu-20251110T171511Z-1-001/MyReweiting/ResultsB0_2022/AntiRadVeto_MC_NoRes_2022_Era1_v0_XGBoost_fom_cut_BDT.root"

vars_gen_to_load = ["gen_cosThetaK", "gen_cosThetaL", "gen_phi", "q2Gen"]
vars_reco_to_load = ["CosThetaK_best", "CosThetaL_best", "Phi_best", "massJ"] 
vars_xgboost_to_load = ["CosThetaK", "CosThetaL", "Phi", "massB_test", "massJ", "TotalWeight"] 

# --- CARGA DE DATOS ---
#Gen NO filt
genNFtr = uproot.open(f_gen)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"1. Gen Non-Filtered (genNFtr) cargado: {len(genNFtr)} eventos")
# Gen Filtered
genFtr = uproot.open(f_gen_filtered)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"2. Gen Filtered (genFtr) cargado: {len(genFtr)} eventos")
# Reco Gen Level
recoGen = uproot.open(f_reco_gen)['ntuple'].arrays(vars_reco_to_load, library='pd')
print(f"3. Reco Gen Level Denom (recoGen) cargado: {len(recoGen)} eventos")
# Final selection 
recoFtr = uproot.open(x_gboost_cut)['treeBd'].arrays(vars_xgboost_to_load, library='pd')
print(f"4. Reco Final (recoFtr) cargado: {len(recoFtr)} eventos")



1. Gen Non-Filtered (genNFtr) cargado: 11589148 eventos
2. Gen Filtered (genFtr) cargado: 307688 eventos
3. Reco Gen Level Denom (recoGen) cargado: 6298017 eventos
4. Reco Final (recoFtr) cargado: 900424 eventos


In [3]:

recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       

RecoGenFlt = recoGen.copy()             
mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()
#2
eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.02, random_state=549)
eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=2)
eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.3, random_state=2)
eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=2)

a1 = np.array(obs_Gen["gen_cosThetaL"])
a2 = np.array(obs_Gen["gen_cosThetaK"])
a3 = np.array(obs_Gen["gen_phi"])

angles = np.array([a1, a2, a3])
valid_observations_mask = ~np.isnan(angles).any(axis=0)
filtered_data = angles[:, valid_observations_mask]

In [4]:
print("Numero de eventes GEN",len(obs_Gen))
print("Numero de eventes GEN Filtered",len(obs_GenFtr))
print("Numero de eventes Reco Gen filtered",len(obs_RecoGenFtr))
print("Numero de eventes Reco Filtered",len(obs_RecoFtr))


Numero de eventes GEN 231783
Numero de eventes GEN Filtered 30769
Numero de eventes Reco Gen filtered 1889406
Numero de eventes Reco Filtered 90043


# NUEVAS FUNVIONES Y PDF

In [5]:
import zfit
from zfit import z
import tensorflow as tf
import numpy as np
import pandas as pd

# PDF física completa
class FullAngular_Physical_PDF(zfit.pdf.BasePDF):
    def __init__(self, obs, FL, S3, S9, AFB, S4, S7, S5, S8, name="FullAngular_Physical_PDF"):
        params = {
            'FL': FL, 'S3': S3, 'S9': S9, 'AFB': AFB,
            'S4': S4, 'S7': S7, 'S5': S5, 'S8': S8
        }
        super().__init__(obs, params, name=name)
    
    def _unnormalized_pdf(self, x):
        vars_list = z.unstack_x(x)
        cos_l = vars_list[0]
        cos_k = vars_list[1]
        phi   = vars_list[2]
        sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 0.0))
        sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 0.0))
        sin2_k = sin_k**2
        cos2_k = cos_k**2
        sin2_l = sin_l**2
        
        cos2l_term = 2.0 * cos_l**2 - 1.0
        sin2l_term = 2.0 * sin_l * cos_l
        sin2k_term = 2.0 * sin_k * cos_k
        
        cos_phi = tf.cos(phi)
        sin_phi = tf.sin(phi)
        cos2_phi = tf.cos(2.0 * phi)
        sin2_phi = tf.sin(2.0 * phi)

        FL = self.params['FL']
        S3 = self.params['S3']
        S9 = self.params['S9']
        AFB = self.params['AFB']
        S4 = self.params['S4']
        S7 = self.params['S7']
        S5 = self.params['S5']
        S8 = self.params['S8']
        
        term1 = 0.75 * (1.0 - FL) * sin2_k
        term2 = FL * cos2_k
        term3 = 0.25 * (1.0 - FL) * sin2_k * cos2l_term
        term4 = -1.0 * FL * cos2_k * cos2l_term
        term5 = S3 * sin2_k * sin2_l * cos2_phi
        term6 = S4 * sin2k_term * sin2l_term * cos_phi
        term7 = S5 * sin2k_term * sin_l * cos_phi
        term8 = (4.0/3.0) * AFB * sin2_k * cos_l
        term9 = S7 * sin2k_term * sin_l * sin_phi
        term10 = S8 * sin2k_term * sin2l_term * sin_phi
        term11 = S9 * sin2_k * sin2_l * sin2_phi
        
        pdf = term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 + term9 + term10 + term11
        return pdf

    @zfit.supports(norm=True)
    def _integrate(self, limits, norm_range, options=None):
        integral_value = 32.0 * np.pi / 9.0
        return tf.constant([integral_value], dtype=self.dtype)


#PDF transformada usando el método Cartesiano No Acotado (Sin degeneraciones)
class FullAngular_Transformed_PDF(zfit.pdf.BasePDF):
    def __init__(self, obs, raw_FL, raw_S3, u1, v1, u2, v2, u3, v3, name="FullAngular_Transformed_PDF"):
        params = {
            'rFL': raw_FL, 'rS3': raw_S3, 
            'u1': u1, 'v1': v1,
            'u2': u2, 'v2': v2,
            'u3': u3, 'v3': v3
        }
        super().__init__(obs, params, name=name)

    def _cartesian_to_physical(self, u, v, R, scale_x, scale_y):
        """
        Mapea los parámetros libres (u, v) al interior del círculo de radio R
        y escala para obtener los observables físicos.
        """
        rho_sq = tf.square(u) + tf.square(v)
        rho = tf.sqrt(tf.maximum(rho_sq, 1e-18)) # Prevenir NaN en la raíz
        
        # Aproximación de Taylor para tanh(rho)/rho cerca del origen para estabilidad numérica
        factor = tf.where(rho_sq < 1e-8, 
                          1.0 - rho_sq / 3.0, 
                          tf.math.tanh(rho) / rho)
        
        x = R * factor * u
        y = R * factor * v
        
        return x * scale_x, y * scale_y

    def _get_physical_coeffs(self):
        # Sector Longitudinal / Transversal Base
        FL = 0.5 * (1.0 + tf.math.tanh(self.params['rFL']))
        FT = 1.0 - FL
        S3 = 0.5 * FT * tf.math.tanh(self.params['rS3'])
    
        # Sector 1: (S9, AFB)
        R1_sq = tf.maximum(0.25 * tf.square(FT) - tf.square(S3), 1e-18)
        R1 = tf.sqrt(R1_sq)
        AFB, S9 = self._cartesian_to_physical(self.params['u1'], self.params['v1'], R1, scale_x=1.5, scale_y=1.0)
    
        # Sector 2: (S4, S7)
        R2_sq = tf.maximum(FL * (FT - 2.0 * S3), 1e-18)
        R2 = tf.sqrt(R2_sq)
        S4, S7 = self._cartesian_to_physical(self.params['u2'], self.params['v2'], R2, scale_x=0.5, scale_y=1.0)
    
        # Sector 3: (S5, S8)
        R3_sq = tf.maximum(FL * (FT + 2.0 * S3), 1e-18)
        R3 = tf.sqrt(R3_sq)
        S5, S8 = self._cartesian_to_physical(self.params['u3'], self.params['v3'], R3, scale_x=1.0, scale_y=0.5)
    
        return FL, S3, S9, AFB, S4, S7, S5, S8

    def _unnormalized_pdf(self, x):
        vars_list = z.unstack_x(x)
        cos_l = vars_list[0]
        cos_k = vars_list[1]
        phi   = vars_list[2]
    
        sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 1e-10))
        sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 1e-10))   
        sin2_k = sin_k**2
        cos2_k = cos_k**2
        sin2_l = sin_l**2
        cos2l_term = 2.0 * cos_l**2 - 1.0
        sin2l_term = 2.0 * sin_l * cos_l
        sin2k_term = 2.0 * sin_k * cos_k
        cos_phi = tf.cos(phi)
        sin_phi = tf.sin(phi)
        cos2_phi = tf.cos(2.0 * phi)
        sin2_phi = tf.sin(2.0 * phi)

        FL, S3, S9, AFB, S4, S7, S5, S8 = self._get_physical_coeffs()
    
        term1 = 0.75 * (1.0 - FL) * sin2_k
        term2 = FL * cos2_k
        term3 = 0.25 * (1.0 - FL) * sin2_k * cos2l_term
        term4 = -1.0 * FL * cos2_k * cos2l_term
        term5 = S3 * sin2_k * sin2_l * cos2_phi
        term6 = S4 * sin2k_term * sin2l_term * cos_phi
        term7 = S5 * sin2k_term * sin_l * cos_phi
        term8 = (4.0/3.0) * AFB * sin2_k * cos_l
        term9 = S7 * sin2k_term * sin_l * sin_phi
        term10 = S8 * sin2k_term * sin2l_term * sin_phi
        term11 = S9 * sin2_k * sin2_l * sin2_phi
    
        pdf = term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 + term9 + term10 + term11
        return pdf

    @zfit.supports(norm=True)
    def _integrate(self, limits, norm_range, options=None):
        integral_value = 32.0 * np.pi / 9.0
        return tf.constant([integral_value], dtype=self.dtype)


def get_inverse_values(phys_params):
    """
    Toma los valores físicos (FL, S3, S9, AFB, S4, S7, S5, S8) y devuelve los 
    parámetros iniciales sin restricciones (rFL, rS3, u1, v1, u2, v2, u3, v3).
    """
    FL, S3, S9, AFB, S4, S7, S5, S8 = phys_params
    def atanh(x): return np.arctanh(np.clip(x, -0.999999, 0.999999))

    rFL = atanh(2.0 * FL - 1.0)
    FT = 1.0 - FL
    rS3 = atanh(S3 / (0.5 * FT))

    def _physical_to_cartesian(x, y, R):
        if R <= 1e-15:
            return 0.0, 0.0
        r_phys = np.sqrt(x**2 + y**2)
        ratio = np.clip(r_phys / R, 0.0, 0.999999)
        rho = np.arctanh(ratio)
        if r_phys < 1e-12:
            return 0.0, 0.0
        factor = rho / r_phys
        return x * factor, y * factor

    # Inversa Sector 1
    R1 = np.sqrt(max(1e-15, 0.25 * FT**2 - S3**2))
    u1, v1 = _physical_to_cartesian((2.0/3.0) * AFB, S9, R1)

    # Inversa Sector 2
    R2 = np.sqrt(max(1e-15, FL * (FT - 2.0 * S3)))
    u2, v2 = _physical_to_cartesian(2.0 * S4, S7, R2)

    # Inversa Sector 3
    R3 = np.sqrt(max(1e-15, FL * (FT + 2.0 * S3)))
    u3, v3 = _physical_to_cartesian(S5, 2.0 * S8, R3)

    return [rFL, rS3, u1, v1, u2, v2, u3, v3]


def apply_transformation_equations(rFL, rS3, u1, v1, u2, v2, u3, v3):
    """
    Aplica las ecuaciones de transformación del espacio cartesiano sin límites AL FISICO.
    """
    FL = 0.5 * (1.0 + np.tanh(rFL))
    FT = 1.0 - FL
    S3 = 0.5 * FT * np.tanh(rS3)

    def _cartesian_to_physical_np(u, v, R, scale_x, scale_y):
        rho = np.sqrt(u**2 + v**2)
        factor = np.tanh(rho) / rho if rho > 1e-8 else 1.0 - (rho**2)/3.0
        x = R * factor * u
        y = R * factor * v
        return x * scale_x, y * scale_y

    # Sector 1
    R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
    AFB, S9 = _cartesian_to_physical_np(u1, v1, R1, 1.5, 1.0)

    # Sector 2
    R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
    S4, S7 = _cartesian_to_physical_np(u2, v2, R2, 0.5, 1.0)

    # Sector 3
    R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
    S5, S8 = _cartesian_to_physical_np(u3, v3, R3, 1.0, 0.5)

    return {'FL': FL, 'S3': S3, 'S9': S9, 'AFB': AFB, 'S4': S4, 'S7': S7, 'S5': S5, 'S8': S8}



def save_phy_results(result_zfit, phys_dict, cov_phys, obs_order, bin_n, base_dir="fit_results", name="fit_results_phys"):
    """
    Guarda los resultados de los observables físicos transformados y su matriz 
    de covarianza propagada en el mismo formato JSON que los resultados del fit.
    """
    output_folder = os.path.join(base_dir, f"{bin_n}")
    os.makedirs(output_folder, exist_ok=True)
    output_file = os.path.join(output_folder, f"{name}.json")
    
    params_dict = {}
    
    errors_phys = np.sqrt(np.diag(cov_phys))
    
    for i, obs_name in enumerate(obs_order):
        val = phys_dict.get(obs_name, 0.0)
        sym_err = errors_phys[i]
        
        # asume errores simétricos NLL "normal"
        params_dict[obs_name] = {
            'value': float(val), 
            'error': float(sym_err), 
            'error_low': float(-sym_err), 
            'error_up': float(sym_err), 
            'error_source': "delta_method"
        }

    cov_list = cov_phys.tolist()
    data_to_save = {
        'bin_index': str(bin_n), 
        'valid': bool(result_zfit.valid), 
        'converged': bool(result_zfit.converged), 
        'fmin': float(result_zfit.fmin), 
        'status': result_zfit.status,
        'parameters': params_dict,
        'covariance': cov_list
    }
    
    with open(output_file, 'w') as f:
        json.dump(data_to_save, f, indent=4)
        
    print(f"[CheckPoint] Resultados físicos (transformados) guardados en: {output_file}")
    return output_file    

     
def plot_nll_profiles(result, params_list, bin_name, base_dir="plots/profiles"):
    bin_dir = os.path.join(base_dir, bin_name)
    os.makedirs(bin_dir, exist_ok=True)
    
    loss = result.loss
    nll_min = result.fmin
    minimizer = zfit.minimize.Minuit(verbosity=0)
    best_fit_values = {p: result.params[p]['value'] for p in params_list}
    
    for param in params_list:
        clean_name = param.name.split('_')[0]
        print(f"--- Calculando perfil NLL para {param.name} ---")
        
        val_opt = best_fit_values[param]
        minos_data = result.params[param].get('minos', {})
        err_lower = minos_data.get('lower', -0.1)
        err_upper = minos_data.get('upper', 0.1)
        
        scan_min = val_opt + 1.5*err_lower
        scan_max = val_opt + 1.5*err_upper
        
        if param.has_limits:
            p_lower = float(param.lower)
            p_upper = float(param.upper)
            epsilon = 1e-4
            
            scan_min = max(scan_min, p_lower + epsilon)
            scan_max = min(scan_max, p_upper - epsilon)
            
        scan_values = np.linspace(scan_min, scan_max, 50)
        nll_values = []
        
        # Escaneo de NLL
        for val_scan in scan_values:
            for p in params_list:
                p.set_value(best_fit_values[p])
            
            param.set_value(val_scan)   
            param.floating = False
            temp_result = minimizer.minimize(loss)
            nll_values.append(temp_result.fmin)


                
        param.floating = True
        nll_values = np.array(nll_values)
        delta_nll = nll_values - nll_min
        
        # ==========================================
        # plots
        # ==========================================
        fig, ax = plt.subplots(figsize=(8, 6))

        ax.plot(scan_values, delta_nll, '-', lw=1, color='blue', label='Profile Likelihood')
        ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label=r'$\Delta$NLL = 0.5 (Errors $1\sigma$)')
        ax.axvline(val_opt, color='green', linestyle='-.', linewidth=1.5, label='Best fit')   

        ax.set_title(f'NLL Profile: {clean_name} ({bin_name})', loc='center', fontsize=14, fontweight='medium', y=1.05)
        ax.set_xlabel(f'{clean_name}', fontsize=16)
        ax.set_ylabel(r'$\Delta$ NLL', fontsize=16)
        ax.set_ylim(bottom=-0.1, top=0.52) 

        hep.cms.label(data=False, loc=0, ax=ax, rlabel="13 TeV", fontname="sans-serif", fontsize=16)

        ax.legend(frameon=False, fontsize=13, loc='upper right') # 'upper center' también suele funcionar bien en perfiles NLL
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='both', which='major', labelsize=14, direction='in', top=True, right=True)

        file_path = f"{bin_dir}/nll_profile_{clean_name}_{bin_name}.png"
        plt.savefig(file_path, dpi=300, bbox_inches='tight')
        plt.close()


def print_polar_physical_variables(rFL, rS3, r1, a1, r2, a2, r3, a3, bin_name=""):
    """
    Calcula e imprime los radios máximos, radios físicos y ángulos 
    reales a partir de los parámetros libres del ajuste.
    """
    #  FL y S3 físicos
    FL = 0.5 * (1.0 + np.tanh(rFL))
    FT = 1.0 - FL
    S3 = 0.5 * FT * np.tanh(rS3)
    
    # calcular los radios
    R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
    r1_phys = (R1 / 2.0) * (1.0 + np.tanh(r1))
    
    R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
    r2_phys = (R2 / 2.0) * (1.0 + np.tanh(r2))
    
    R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
    r3_phys = (R3 / 2.0) * (1.0 + np.tanh(r3))
    
    print("\n" + "="*60)
    print(f"VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - {bin_name}")
    print("="*60)
    
    print(f"[Sector 1: S9, AFB]")
    print(f"  Radio máximo (R1)     = {R1:.6f}")
    print(f"  Radio físico (r1_phys)= {r1_phys:.6f}")
    print(f"  Ángulo (alpha_1)      = {a1:.6f} rad")
    
    print(f"\n[Sector 2: S4, S7]")
    print(f"  Radio máximo (R2)     = {R2:.6f}")
    print(f"  Radio físico (r2_phys)= {r2_phys:.6f}")
    print(f"  Ángulo (alpha_2)      = {a2:.6f} rad")
    
    print(f"\n[Sector 3: S5, S8]")
    print(f"  Radio máximo (R3)     = {R3:.6f}")
    print(f"  Radio físico (r3_phys)= {r3_phys:.6f}")
    print(f"  Ángulo (alpha_3)      = {a3:.6f} rad")
    print("="*60 + "\n")

# Gen  FIT Transformation Cartesian 

In [6]:
#%%capture data_gen_fit_transformed_polar
q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0],"bin7":[11.0, 12.5], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}
#q2_bins = { "bin10":[17.0, 23.0]}

def get_physical_array(r_array):
    phys_dict = apply_transformation_equations(*r_array)
    return np.array([phys_dict.get(key, 0.0) for key in obs_order])

def compute_jacobian(func, x, epsilon=1e-5):
    n_in = len(x)
    y_center = func(x)
    n_out = len(y_center)
    
    J = np.zeros((n_out, n_in))
    
    for i in range(n_in):
        x_plus = np.copy(x)
        x_minus = np.copy(x)
        
        x_plus[i] += epsilon
        x_minus[i] -= epsilon
        
        y_plus = func(x_plus)
        y_minus = func(x_minus)
        
        J[:, i] = (y_plus - y_minus) / (2.0 * epsilon)
        
    return J


for binN in q2_bins.keys():
    print(f"\n{'='*60}\nProcesando {binN} con rango q2: {q2_bins[binN]}\n{'='*60}")
    json_path = f"fit_results/gen/gen_phy_slsqp/{binN}/fit_results_gen_physical_slsqp_{binN}.json"
    
    # init_rFL, init_rS3 = 0.5, 0.01
    # init_u1, init_v1 = 0.01, 0.01
    # init_u2, init_v2 = 0.01, 0.01
    # init_u3, init_v3 = 0.01, 0.01
    

    if os.path.exists(json_path):
        print(f"-> Leyendo semillas iniciales de: {json_path}")
        with open(json_path, 'r') as file:
            data = json.load(file)
        params_dict = data.get("parameters", {})
        
        phys_params_list = [
            params_dict["FL"]["value"],
            params_dict["S3"]["value"],
            params_dict["S9"]["value"],
            params_dict["AFB"]["value"],
            params_dict["S4"]["value"],
            params_dict["S7"]["value"],
            params_dict["S5"]["value"],
            params_dict["S8"]["value"]
        ]
        
        inv_vals = get_inverse_values(phys_params_list)
        print(inv_vals)
        init_rFL, init_rS3, init_u1, init_v1, init_u2, init_v2, init_u3, init_v3 = inv_vals = inv_vals
    
        

    else:
        print(f"-> ADVERTENCIA: No se encontró el archivo {json_path}")

    
    # ======================================================
    # CONFIGURACIÓN DEL ESPACIO 
    # ======================================================
    cos_l = zfit.Space('cos_l', limits=(-1, 1))
    cos_k = zfit.Space('cos_k', limits=(-1, 1))
    phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
    obs_ang = cos_l * cos_k * phi  

    # ======================================================
    # CREACIÓN DE PARÁMETROS ZFIT CON LAS SEMILLAS DINÁMICAS
    # ======================================================
    limit = 9
    # F_L y S3
    rFL = zfit.Parameter(f'rFL_{binN}', init_rFL, step_size=1e-1, lower_limit=-limit, upper_limit=limit)
    rS3 = zfit.Parameter(f'rS3_{binN}', init_rS3, step_size=1e-1, lower_limit=-limit, upper_limit=limit)
    # S9 y AFB (Sector 1)
    u1 = zfit.Parameter(f'u1_{binN}', init_u1, step_size=1e-1, lower_limit=-limit, upper_limit=limit)
    v1 = zfit.Parameter(f'v1_{binN}', init_v1, step_size=1e-1, lower_limit=-limit, upper_limit=limit)

    # S4 y S7 (Sector 2)
    u2 = zfit.Parameter(f'u2_{binN}', init_u2, step_size=1e-1, lower_limit=-limit, upper_limit=limit)
    v2 = zfit.Parameter(f'v2_{binN}', init_v2, step_size=1e-1, lower_limit=-limit, upper_limit=limit)

    # S5 y S8 (Sector 3)
    u3 = zfit.Parameter(f'u3_{binN}', init_u3, step_size=1e-1, lower_limit=-limit, upper_limit=limit)
    v3 = zfit.Parameter(f'v3_{binN}', init_v3, step_size=1e-1, lower_limit=-limit, upper_limit=limit)


    # # F_L y S3
    # rFL = zfit.Parameter(f'rFL_{binN}', init_rFL, step_size=1e-3)
    # rS3 = zfit.Parameter(f'rS3_{binN}', init_rS3, step_size=1e-3)
    # # S9 y AFB (Sector 1)
    # u1 = zfit.Parameter(f'u1_{binN}', init_u1, step_size=1e-3)
    # v1 = zfit.Parameter(f'v1_{binN}', init_v1, step_size=1e-3)
    # # S4 y S7 (Sector 2)
    # u2 = zfit.Parameter(f'u2_{binN}', init_u2, step_size=1e-3)
    # v2 = zfit.Parameter(f'v2_{binN}', init_v2, step_size=1e-3)
    # # S5 y S8 (Sector 3)
    # u3 = zfit.Parameter(f'u3_{binN}', init_u3, step_size=1e-3)
    # v3 = zfit.Parameter(f'v3_{binN}', init_v3, step_size=1e-3)


    fit_params_list = [rFL, rS3, u1, v1, u2, v2, u3, v3]
    pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, u1, v1, u2, v2, u3, v3)
    # ======================================================
    # PDFs Y CARGA DE DATOS
    # ======================================================
    
    obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
    print("NÚMERO DE EVENTOS:", len(obs_Gen_q2),"\n")
    data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)

    # ======================================================
    # FITS
    # ======================================================

    print("\n" + "="*60)
    print(">>> INICIANDO FIT GEN LEVEL TRANSFORMED SPACE <<<")
    print("="*60)
    result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
    save_fit_results(result_gen, binN, base_dir="fit_results/gen_transformed_polar_cartesian", name=f"fit_results_gen_physical_{binN}")
    r_values = np.array([result_gen.params[p]['value'] for p in fit_params_list])
    cov_fit = result_gen.covariance(params=fit_params_list)
    print(result_gen.params)
    print_polar_physical_variables(*r_values, bin_name=binN)
    obs_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']
    J = compute_jacobian(get_physical_array, r_values)
    cov_phys = J @ cov_fit @ J.T
    errors_phys = np.sqrt(np.diag(cov_phys))

    phys_vals_gen_dict = apply_transformation_equations(*r_values)
    save_phy_results(result_zfit=result_gen, phys_dict=phys_vals_gen_dict, cov_phys=cov_phys, obs_order=obs_order, bin_n=binN, base_dir="fit_results/gen_transformed_polar_cartesian", name=f"fit_results_gen_transformed_phys_{binN}")
    print("\n" + "-"*80)
    print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN})")
    print("-"*80)
    print(f"{'Observable':<10} | {'Valor Físico':<15} | {'Error':<25}")
    print("-"*80)

    for i, key in enumerate(obs_order):
        val = phys_vals_gen_dict.get(key, 0.0)
        err = errors_phys[i]
        print(f"{key:<10} | {val:>15.6f} | +/- {err:<15.6f}")
    print("-"*80 + "\n")
    #plot_nll_profiles(result_gen, fit_params_list, binN, base_dir="plots/profiles_cartesian")
    save_correlation_matrix(result_gen, fit_params_list, binN, out_dir=f"fit_results/gen/gen_transformed_polar_cartesian/{binN}")




Procesando bin1 con rango q2: [1.1, 2.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin1/fit_results_gen_physical_slsqp_bin1.json
[0.37139004652924373, 0.006349338387853855, -0.01017993372573161, -0.053297872350286796, 0.12553425548141198, 0.027530138126609954, -0.022414265361949553, -0.04358139949825023]
NÚMERO DE EVENTOS: 7975 


>>> INICIANDO FIT GEN LEVEL TRANSFORMED SPACE <<<


2026-03-16 23:08:52.038489: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-16 23:08:52.109045: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-16 23:08:52.113466: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

[CheckPoint] Resultados guardados en: fit_results/gen_transformed_polar_cartesian/bin1/fit_results_gen_physical_bin1.json
name        value  (rounded)                minos    at limit
--------  ------------------  -------------------  ----------
rFL_bin1            0.372385  -  0.017   +  0.017       False
rS3_bin1            0.047111  -  0.064   +  0.065       False
u1_bin1           -0.0353815  -  0.029   +  0.029       False
v1_bin1           -0.0220431  -  0.065   +  0.065       False
u2_bin1            0.0615836  -  0.054   +  0.055       False
v2_bin1            0.0108934  -  0.027   +  0.027       False
u3_bin1            0.0269834  -  0.026   +  0.026       False
v3_bin1           -0.0767576  -  0.053   +  0.052       False

VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - bin1
[Sector 1: S9, AFB]
  Radio máximo (R1)     = 0.160803
  Radio físico (r1_phys)= 0.077558
  Ángulo (alpha_1)      = -0.022043 rad

[Sector 2: S4, S7]
  Radio máximo (R2)     = 0.456098
  Radio físic

In [7]:
# with open('data_gen_fit_transformed_polar.txt', 'w') as archivo:
#     archivo.write(data_gen_fit_transformed_polar.stdout) 
#     archivo.write(data_gen_fit_transformed_polar.stderr)


In [8]:
ñllñl

NameError: name 'ñllñl' is not defined

# Fit Efficiency

In [ ]:
#%%capture mi_registro
# q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0],"bin7":[11.0, 12.5], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}

q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0]}



for binN in q2_bins.keys():
    print(f"\n{'='*60}\nProcesando {binN} con rango q2: {q2_bins[binN]}\n{'='*60}")

    # ======================================================
    # CONFIGURACIÓN DEL ESPACIO 
    # ======================================================
    cos_l = zfit.Space('cos_l', limits=(-1, 1))
    cos_k = zfit.Space('cos_k', limits=(-1, 1))
    phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
    obs_ang = cos_l * cos_k * phi  
    init_rFL, init_rS3 = 0.5, 0.01
    init_u1, init_v1 = 0.01, 0.01
    init_u2, init_v2 = 0.01, 0.01
    init_u3, init_v3 = 0.01, 0.01
    lim_val = 5
    # Parámetros Físicos con límites
    # rFL  = zfit.Parameter(f'rFL_{binN}_reco',  0.5)
    # rS3  = zfit.Parameter(f'rS3_{binN}_reco',  0.0, step_size=0.0000001, lower_limit=-lim_val, upper_limit=lim_val)
    # rS9  = zfit.Parameter(f'rS9_{binN}_reco',  0.0, step_size=0.0000001, lower_limit=-lim_val, upper_limit=lim_val)
    # rAFB = zfit.Parameter(f'rAFB_{binN}_reco', 0.0, step_size=0.0000001, lower_limit=-lim_val, upper_limit=lim_val)
    # rS4  = zfit.Parameter(f'rS4_{binN}_reco',  0.0, step_size=0.0000001, lower_limit=-lim_val, upper_limit=lim_val)
    # rS7  = zfit.Parameter(f'rS7_{binN}_reco',  0.0, step_size=0.000001, lower_limit=-lim_val, upper_limit=lim_val)
    # rS5  = zfit.Parameter(f'rS5_{binN}_reco',  1e-3, step_size=1e-6, lower_limit=-lim_val, upper_limit=lim_val)
    # rS8  = zfit.Parameter(f'rS8_{binN}_reco',  0.0, step_size=0.0000001, lower_limit=-lim_val, upper_limit=lim_val)

    limit = 10
    # F_L y S3
    rFL = zfit.Parameter(f'rFL_{binN}', init_rFL, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    rS3 = zfit.Parameter(f'rS3_{binN}', init_rS3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    # S9 y AFB (Sector 1)
    u1 = zfit.Parameter(f'u1_{binN}', init_u1, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    v1 = zfit.Parameter(f'v1_{binN}', init_v1, step_size=1e-3, lower_limit=-limit, upper_limit=limit)

    # S4 y S7 (Sector 2)
    u2 = zfit.Parameter(f'u2_{binN}', init_u2, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    v2 = zfit.Parameter(f'v2_{binN}', init_v2, step_size=1e-3, lower_limit=-limit, upper_limit=limit)

    # S5 y S8 (Sector 3)
    u3 = zfit.Parameter(f'u3_{binN}', init_u3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    v3 = zfit.Parameter(f'v3_{binN}', init_v3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    # Listas auxiliares
    #r_keys = ['rFL', 'rS3', 'rS9', 'rAFB', 'rS4', 'rS7', 'rS5', 'rS8']
    fit_params_list = [rFL, rS3, u1, v1, u2, v2, u3, v3]
    pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, u1, v1, u2, v2, u3, v3)
    
    # PDF de Eficiencia
    coef_acc, nx_acc, ny_acc = load_bernstein_model(f"models/{binN}/acc_gen_model_{binN}.json")
    coef_acc_phi, n_phi_acc = load_bernstein1d_model(f"models/{binN}/acc_gen_model_phi_{binN}.json")

    coef_reco, nx_reco, ny_reco = load_bernstein_model(f"models/{binN}/eff_reco_model_{binN}.json")
    coef_reco_phi, n_phi_reco = load_bernstein1d_model(f"models/{binN}/eff_reco_model_phi_{binN}.json")

    eff_pdf = Efficiency_Bernstein_Factorized(
        obs=obs_ang, coef_acc_2d=coef_acc, coef_acc_phi=coef_acc_phi, nx_acc=nx_acc, ny_acc=ny_acc, n_phi_acc=n_phi_acc,
        coef_reco_2d=coef_reco, coef_reco_phi=coef_reco_phi,nx_reco=nx_reco, ny_reco=ny_reco, n_phi_reco=n_phi_reco, name=f"Eff_Model_{binN}")
    pdf_sig = zfit.pdf.ProductPDF([pdf_ang_trans, eff_pdf])


    # Carga de Datos
    obs_RecoFtr_q2 = select_q2_bin(obs_RecoFtr, binN, "q2")
    data_reco = zfit.Data.from_numpy(array=obs_RecoFtr_q2[["CosThetaL", "CosThetaK", "Phi"]].to_numpy(), obs=obs_ang)
    
    # ======================================================
    # FITS
    # ======================================================
    # print("\n" + "="*60)
    # print(">>> INICIANDO FIT GEN LEVEL (CONTROL)")
    # print("="*60)
    # result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
    # print(result_gen.params)
    # results_gen_save = save_fit_results(result_gen, binN, base_dir="fit_results/gen", name=f"fit_results_gen_transformed_{binN}")
    # r_values = [result_gen.params[p]['value'] for p in fit_params_list]
    # phys_vals_gen = apply_transformation_equations(*r_values)
    # print("\n" + "-"*60)
    # print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN})")
    # print("-"*60)
    # print(f"{'Observable':<10} | {'Valor Físico':<15}")
    # print("-"*60)
    # print_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']    
    # for key in print_order:
    #     val = phys_vals_gen.get(key, 0.0)
    #     print(f"{key:<10} | {val:>15.6f}")
    # print("-"*60 + "\n")




    print("\n" + "="*60)
    print(">>> INICIANDO FIT RECO)")
    print("="*60)
    for p in fit_params_list: p.set_value(0.01)    
    # pdf_sig.update_integration_options(max_draws=200000, tol=1e-5)
    result_reco, errors_reco = run_fit(pdf_sig, data_reco)
    print("\n" + "="*60)
    print(">>> RESULTADOS FINALES DEL FIT RECO")
    print("="*60)
    print(result_reco)
    results_save = save_fit_results(result_reco, binN, base_dir="fit_results/reco")

    # callculo y tabla de Observables Físicos para RECO
    r_values_reco = [result_reco.params[p]['value'] for p in fit_params_list]
    phys_vals_reco = apply_transformation_equations(*r_values_reco)
    print("\n" + "-"*60)
    print(f"RESUMEN DE OBSERVABLES FÍSICOS RECO (Bin: {binN})")
    print("-"*60)
    print(f"{'Observable':<10} | {'Valor Físico':<15}")
    print("-"*60)
    print_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']    
    for key in print_order:
        val = phys_vals_reco.get(key, 0.0)
        print(f"{key:<10} | {val:>15.6f}")
    print("-"*60 + "\n")
    # plot_nll_profiles(result_reco, fit_params_list, binN,base_dir="plots/reco/profiles_transform_v1")
    save_correlation_matrix(result_reco, fit_params_list, binN, out_dir=f"plots/reco/correlation_matrix/transformation_1")

    


Procesando bin1 con rango q2: [1.1, 2.0]


FileNotFoundError: [Errno 2] No such file or directory: 'models/bin1/acc_gen_model_bin1.json'